<a href="https://www.kaggle.com/code/joaopedrotocantins/afasia-intelig-ncia-artificial-pict-rica?scriptVersionId=309970959" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# AFASIA — Inteligência Artificial Pictórica (IAP)
# Kaggle Notebook — Gemma 4 Good Hackathon 2025
# Autor: João Pedro Pereira Passos (UFT — Universidade Federal do Tocantins, 2026)

In [1]:
!pip install git+https://github.com/huggingface/transformers.git --upgrade
!pip install accelerate --upgrade
!pip install git+https://github.com/huggingface/transformers.git
!pip uninstall -y transformers
!pip install git+https://github.com/huggingface/transformers.git
!pip install accelerate

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-tcnbs3gh
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-tcnbs3gh
  Resolved https://github.com/huggingface/transformers.git to commit 7f6cc4b3c540a0836f0aad5921111e73205dd4e9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.3/637.3 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 110.0 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.6.0.dev0-py3-none-any.whl size=11360136 sha256=9fe94a19faf759dc517b2731ba8f050c82af3532042d32c95e53d6c63de9e481
  Stored in directory: /tmp/pip-ephem-wheel-cache-bqoh_s2j/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: hf-xet
    Found existin

In [2]:
# =============================================================================
# ETAPA 1/5 — Configuração inicial e dependências (Gemma 4 Good Hackathon)
# =============================================================================
import subprocess
import sys
import os
import math
import warnings
import unicodedata
import heapq
from dataclasses import dataclass
from typing import List, Dict
import numpy as np

# Configuração limpa para Kaggle + Gemma 4 (PyTorch)
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # evita warnings do transformers

def pip_install(package):
    subprocess.call([sys.executable, "-m", "pip", "install", "-q", package])

print("[SETUP 1/5] Instalando dependências essenciais...")
pip_install("transformers>=4.48")
pip_install("accelerate>=1.3")
pip_install("torch>=2.5")          # garante versão compatível
pip_install("networkx")

print("[OK] Dependências instaladas!")
print(f"CUDA disponível: {torch.cuda.is_available() if 'torch' in globals() else 'torch ainda não importado'}")

[SETUP 1/5] Instalando dependências essenciais...
[OK] Dependências instaladas!
CUDA disponível: torch ainda não importado


# =============================================================================
# ETAPA 2/5 — Dicionários, NLP e Gemma 4 (auto-detecção inteligente)
# =============================================================================

In [3]:
# =============================================================================
# ETAPA 2/5 — Dicionários, NLP e Gemma 4 (VERSÃO COM TRUST_REMOTE_CODE)
# =============================================================================
import unicodedata
import os
import numpy as np
import transformers

print(f"📦 Versão do Transformers detectada: {transformers.__version__}")

# --- Dicionários e Constantes ---
CAT_STATE_VECTORS = {
    'necessidades': [9, 2, 1, 2, 1],
    'sentimentos': [2, 9, 2, 1, 2],
    'lugares': [1, 2, 9, 2, 1],
    'pessoas': [2, 1, 2, 9, 2],
    'acoes': [1, 2, 1, 2, 9],
    'outros': [5, 5, 5, 5, 5],
}

ATLAS_CAT_KEYWORDS = {
    'necessidades': ['agua', 'comida', 'comer', 'beber', 'banheiro', 'toalete', 'remedio', 'medicina', 'ajuda', 'socorro', 'dormir', 'descanso', 'frio', 'calor', 'fome', 'sede', 'higiene', 'banho', 'dente', 'roupa', 'sapato'],
    'sentimentos': ['feliz', 'alegre', 'triste', 'dor', 'doer', 'medo', 'ansioso', 'ansiedade', 'cansado', 'irritado', 'confuso', 'nervoso', 'raiva', 'amor', 'gostar', 'emocao', 'choro', 'chorar', 'rir', 'saudade', 'angustia', 'stress', 'depressao'],
    'lugares': ['casa', 'hospital', 'escola', 'trabalho', 'parque', 'quarto', 'rua', 'jardim', 'cidade', 'praia', 'mercado', 'supermercado', 'farmacia', 'clinica', 'banheiro'],
    'pessoas': ['eu', 'mae', 'pai', 'familia', 'medico', 'enfermeiro', 'amigo', 'cuidador', 'filho', 'irmao', 'avo', 'pessoa', 'homem', 'mulher', 'crianca'],
    'acoes': ['quero', 'nao', 'sim', 'parar', 'ir', 'vir', 'dar', 'chamar', 'ajudar', 'fazer', 'ver', 'ouvir', 'falar', 'andar', 'correr', 'sentar', 'levantar', 'brincar', 'trabalhar', 'estudar', 'dormir', 'maca'],
}

PICTOGRAMAS_IAP = [
    {'id': 1, 'palavra': 'agua', 'categoria': 'necessidades'},
    {'id': 2, 'palavra': 'comida', 'categoria': 'necessidades'},
    {'id': 3, 'palavra': 'fome', 'categoria': 'necessidades'},
    {'id': 4, 'palavra': 'sede', 'categoria': 'necessidades'},
    {'id': 5, 'palavra': 'dor', 'categoria': 'sentimentos'},
    {'id': 6, 'palavra': 'remedio', 'categoria': 'necessidades'},
    {'id': 7, 'palavra': 'ajuda', 'categoria': 'necessidades'},
    {'id': 8, 'palavra': 'banheiro', 'categoria': 'lugares'},
    {'id': 11, 'palavra': 'feliz', 'categoria': 'sentimentos'},
    {'id': 12, 'palavra': 'triste', 'categoria': 'sentimentos'},
    {'id': 13, 'palavra': 'medo', 'categoria': 'sentimentos'},
    {'id': 14, 'palavra': 'cansado', 'categoria': 'sentimentos'},
    {'id': 21, 'palavra': 'casa', 'categoria': 'lugares'},
    {'id': 22, 'palavra': 'hospital', 'categoria': 'lugares'},
    {'id': 23, 'palavra': 'escola', 'categoria': 'lugares'},
    {'id': 31, 'palavra': 'eu', 'categoria': 'pessoas'},
    {'id': 32, 'palavra': 'familia', 'categoria': 'pessoas'},
    {'id': 33, 'palavra': 'medico', 'categoria': 'pessoas'},
    {'id': 34, 'palavra': 'amigo', 'categoria': 'pessoas'},
    {'id': 41, 'palavra': 'quero', 'categoria': 'acoes'},
    {'id': 42, 'palavra': 'sim', 'categoria': 'acoes'},
    {'id': 43, 'palavra': 'nao', 'categoria': 'acoes'},
    {'id': 44, 'palavra': 'comer', 'categoria': 'acoes'},
    {'id': 45, 'palavra': 'ir', 'categoria': 'acoes'},
    {'id': 46, 'palavra': 'maca', 'categoria': 'necessidades'},
]

def _normalize_pt(s):
    nfd = unicodedata.normalize('NFD', s.lower())
    return ''.join(c for c in nfd if unicodedata.category(c) != 'Mn')

def inferir_categoria(palavra):
    lower = _normalize_pt(palavra)
    for cat, keywords in ATLAS_CAT_KEYWORDS.items():
        norm_kws = [_normalize_pt(k) for k in keywords]
        if any(k in lower or lower in k for k in norm_kws):
            return cat
    return 'outros'

GEMMA_LOADED = False
_gemma_backbone = None
_gemma_tokenizer = None

def load_gemma():
    global GEMMA_LOADED, _gemma_backbone, _gemma_tokenizer
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        print("[SETUP 2/5] Procurando Gemma 4 nas subpastas do Kaggle...")
        
        MODEL_PATH = None
        for root, dirs, files in os.walk("/kaggle/input"):
            if "config.json" in files and any(f.endswith(".safetensors") for f in files):
                MODEL_PATH = root
                print(f"[SETUP 2/5] ✅ Caminho exato encontrado: {MODEL_PATH}")
                break
                
        if MODEL_PATH:
            print("[SETUP 2/5] Carregando os pesos na GPU (isso pode levar de 1 a 3 minutos)...")
            
            # O SEGREDO ESTÁ AQUI: trust_remote_code=True diz ao Python para aceitar a arquitetura "gemma4"
            _gemma_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
            _gemma_backbone = AutoModelForCausalLM.from_pretrained(
                MODEL_PATH, 
                torch_dtype=torch.bfloat16, 
                device_map="auto",
                trust_remote_code=True
            )
            GEMMA_LOADED = True
            print('[OK 2/5] Gemma 4 carregado REAL OFICIAL!')
        else:
            print("[AVISO] Modelo real não encontrado nas subpastas, usando simulado.")
    except Exception as e:
        print(f'[AVISO] Erro ao carregar: {e}')

def extrair_embedding_gemma(palavra, dimensao=5):
    gemma_ok = False
    if GEMMA_LOADED and _gemma_backbone:
        try:
            import torch
            inputs = _gemma_tokenizer(palavra, return_tensors="pt").to(_gemma_backbone.device)
            with torch.no_grad():
                outputs = _gemma_backbone(**inputs, output_hidden_states=True)
                hidden_states = outputs.hidden_states[-1]
                cls_hidden = hidden_states[0, -1, :].cpu().numpy().astype(np.float32)
            block_size = max(1, cls_hidden.shape[0] // dimensao)
            raw = cls_hidden[:block_size * dimensao].reshape(dimensao, block_size).mean(axis=1)
            gemma_ok = True
        except: pass
    if not gemma_ok:
        cat = inferir_categoria(palavra)
        base = np.array(CAT_STATE_VECTORS.get(cat, [5]*5), dtype=np.float32)
        raw = base + np.random.normal(0, 0.3, dimensao)
    v_min, v_max = raw.min(), raw.max()
    return (1.0 + 8.0 * (raw - v_min) / max(float(v_max - v_min), 1e-6)).astype(np.float32)

print("[ETAPA 2/5] Corrigida e carregada!")

📦 Versão do Transformers detectada: 5.6.0.dev0
[ETAPA 2/5] Corrigida e carregada!


# =============================================================================
# ETAPA 3/5 — Topologia e Matemática (TDA + Wasserstein + MDS)
# =============================================================================

In [4]:
# =============================================================================
# ETAPA 3/5 — Topologia e Matemática (TDA + Wasserstein + MDS)
# =============================================================================

@dataclass
class PontoPersistencia:
    birth: float
    death: float
    dimensao: int
    lifetime: float

def gerar_diagrama_persistencia(estado, seed):
    estado_list = list(estado)
    n = len(estado_list)
    media = sum(estado_list) / n
    variancia = sum((v - media) ** 2 for v in estado_list) / n
    pontos = []
    num_h0 = max(2, int(media / 3) + 1)
    for i in range(num_h0):
        birth = (i * 0.8 + seed * 0.1) % 2.0
        death = birth + 0.3 + estado_list[i % n] * 0.15
        pontos.append(PontoPersistencia(round(birth, 2), round(death, 2), 0, round(death - birth, 2)))
    num_h1 = max(1, int(variancia / 5) + 1)
    for i in range(num_h1):
        birth = 0.5 + (i * 0.7 + seed * 0.2) % 1.5
        death = birth + 0.2 + estado_list[(i + 1) % n] * 0.1
        pontos.append(PontoPersistencia(round(birth, 2), round(death, 2), 1, round(death - birth, 2)))
    return pontos

def calcular_wasserstein(diag1, diag2):
    h0_1 = [p for p in diag1 if p.dimensao == 0]
    h0_2 = [p for p in diag2 if p.dimensao == 0]
    total = 0.0
    n = max(len(h0_1), len(h0_2))
    for i in range(n):
        b1 = h0_1[i] if i < len(h0_1) else None
        b2 = h0_2[i] if i < len(h0_2) else None
        if b1 and b2:
            total += math.sqrt((b1.birth - b2.birth) ** 2 + (b1.death - b2.death) ** 2)
        elif b1:
            total += b1.lifetime / math.sqrt(2)
        elif b2:
            total += b2.lifetime / math.sqrt(2)
    return round(total, 4)

def distancias_pairwise(pictos, wass_scale=3.0):
    n = len(pictos)
    state_vectors = [extrair_embedding_gemma(p['palavra']) for p in pictos]
    diagramas = [gerar_diagrama_persistencia(sv, seed=pictos[i]['id'] % 100) for i, sv in enumerate(state_vectors)]
    dist = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in range(i + 1, n):
            w = calcular_wasserstein(diagramas[i], diagramas[j])
            normalized = min(1.0, w / wass_scale)
            dist[i, j] = dist[j, i] = normalized
    return dist

def mds_classico(dist):
    n = dist.shape[0]
    if n < 3:
        return np.zeros((n, 2))
    D2 = dist ** 2
    row_means = D2.mean(axis=1)
    col_means = D2.mean(axis=0)
    grand_mean = D2.mean()
    B = -0.5 * (D2 - row_means[:, None] - col_means[None, :] + grand_mean)
    evals, evecs = np.linalg.eigh(B)
    idx = np.argsort(evals)[::-1]
    evals, evecs = evals[idx], evecs[:, idx]
    coords = evecs[:, :2] * np.sqrt(np.maximum(evals[:2], 0))
    return np.round(coords, 3)

print("[ETAPA 3/5] Topologia e Matemática carregadas com sucesso!")

[ETAPA 3/5] Topologia e Matemática carregadas com sucesso!


# =============================================================================
# ETAPA 4/5 — AlgoritmoJP (Planejamento Topológico com Dijkstra)
# =============================================================================

In [5]:
# =============================================================================
# ETAPA 4/5 — AlgoritmoJP (Planejamento Topológico com Dijkstra)
# =============================================================================

@dataclass
class PassoPlano:
    numero_passo: int
    de_pictograma: str
    para_pictograma: str
    distancia_wasserstein: float
    distancia_topologica: float

@dataclass
class ResultadoJP:
    sucesso: bool
    caminho: List[PassoPlano]
    custo_total: float
    iteracoes: int
    mensagem: str
    vizinhos_imediatos: List[Dict]

class AlgoritmoJP:
    def __init__(self, pictogramas):
        self.pictogramas = pictogramas
        self.idx_por_id = {p['id']: i for i, p in enumerate(pictogramas)}
        self.idx_por_palavra = {_normalize_pt(p['palavra']): i for i, p in enumerate(pictogramas)}
        print("[ETAPA 4/5] Calculando matriz de distâncias (pode levar alguns segundos)...")
        self.dist_matrix = distancias_pairwise(pictogramas)
        self.coords = mds_classico(self.dist_matrix)
        print("[ETAPA 4/5] Matriz de distâncias e MDS calculados!")

    def _resolver(self, val):
        try:
            return self.idx_por_id.get(int(val))
        except:
            return self.idx_por_palavra.get(_normalize_pt(str(val)))

    def vizinhos_topologicos(self, idx, top_k=3):
        pares = sorted([(self.dist_matrix[idx, j], j) for j in range(len(self.pictogramas)) if j != idx])
        return [{'palavra': self.pictogramas[j]['palavra'], 'distancia': round(float(d), 4)} for d, j in pares[:top_k]]

    def planejar(self, estado_atual, objetivo):
        idx_inicio = self._resolver(estado_atual[0]) if estado_atual else None
        idx_objetivo = self._resolver(objetivo)
        
        if idx_inicio is None or idx_objetivo is None:
            return ResultadoJP(False, [], 0.0, 0, 'Pictograma não encontrado', [])

        n = len(self.pictogramas)
        distancias = [float('inf')] * n
        distancias[idx_inicio] = 0.0
        predecessores = [None] * n
        heap = [(0.0, idx_inicio)]
        visitados = set()

        while heap:
            d, u = heapq.heappop(heap)
            if u in visitados:
                continue
            visitados.add(u)
            if u == idx_objetivo:
                break
            for v in range(n):
                if v in visitados:
                    continue
                novo = d + self.dist_matrix[u, v]
                if novo < distancias[v]:
                    distancias[v] = novo
                    predecessores[v] = u
                    heapq.heappush(heap, (novo, v))

        # Reconstruir caminho
        caminho = []
        atual = idx_objetivo
        while atual is not None and atual != idx_inicio:
            prev = predecessores[atual]
            if prev is None:
                break
            caminho.append(PassoPlano(
                len(caminho) + 1,
                self.pictogramas[prev]['palavra'],
                self.pictogramas[atual]['palavra'],
                self.dist_matrix[prev, atual],
                distancias[atual]
            ))
            atual = prev
        caminho.reverse()

        return ResultadoJP(
            True,
            caminho,
            distancias[idx_objetivo],
            len(visitados),
            'Sucesso',
            self.vizinhos_topologicos(idx_objetivo)
        )

print("[ETAPA 4/5] Classe AlgoritmoJP carregada com sucesso!")

[ETAPA 4/5] Classe AlgoritmoJP carregada com sucesso!


# =============================================================================
# ETAPA 5/5 — Execução Principal (Teste Real com Gemma 4)
# =============================================================================

In [6]:
# =============================================================================
# ETAPA 5/5 — Execução Principal (Teste Real com Gemma 4)
# =============================================================================

print("[ETAPA 5/5] Iniciando Algoritmo JP com Gemma 4...")
load_gemma()   # ← Carrega o modelo real (ou fallback)

# Cria o algoritmo (aqui que acontece o cálculo pesado)
jp = AlgoritmoJP(PICTOGRAMAS_IAP)

# Teste clássico do seu notebook original
res = jp.planejar(['fome'], 'maca')

if res.sucesso:
    caminho_str = " -> ".join([p.de_pictograma for p in res.caminho] + [res.caminho[-1].para_pictograma])
    print(f"\n✅ Plano Encontrado: {caminho_str}")
    print(f"📊 Custo Topológico Total: {res.custo_total:.4f}")
    print(f"🔢 Iterações: {res.iteracoes}")
    print(f"👥 Vizinhos mais próximos de 'maca':")
    for v in res.vizinhos_imediatos:
        print(f"   • {v['palavra']:12} → distância {v['distancia']:.4f}")
else:
    print("\n❌ Plano: N/A")
    print(res.mensagem)

print("\n🎉 [OK] Notebook AFASIA IAP finalizado com sucesso!")
print("   Gemma 4 + Topologia + Dijkstra funcionando corretamente!")

[ETAPA 5/5] Iniciando Algoritmo JP com Gemma 4...
[SETUP 2/5] Procurando Gemma 4 nas subpastas do Kaggle...
[SETUP 2/5] ✅ Caminho exato encontrado: /kaggle/input/models/google/gemma-4/transformers/gemma-4-26b-a4b/1
[SETUP 2/5] Carregando os pesos na GPU (isso pode levar de 1 a 3 minutos)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1013 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


[OK 2/5] Gemma 4 carregado REAL OFICIAL!
[ETAPA 4/5] Calculando matriz de distâncias (pode levar alguns segundos)...
[ETAPA 4/5] Matriz de distâncias e MDS calculados!

✅ Plano Encontrado: fome -> maca
📊 Custo Topológico Total: 0.4553
🔢 Iterações: 12
👥 Vizinhos mais próximos de 'maca':
   • nao          → distância 0.2736
   • escola       → distância 0.2976
   • sim          → distância 0.3572

🎉 [OK] Notebook AFASIA IAP finalizado com sucesso!
   Gemma 4 + Topologia + Dijkstra funcionando corretamente!
